# 13 The Medallion Architecture with Delta Lake

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Build a small but realistic Bronze → Silver → Gold pipeline that uses the Delta features from this folder: append-only Bronze with audit columns, a Silver table maintained with <code>MERGE</code> and protected by constraints, change data feed from Silver, and a Gold table updated <b>incrementally</b> from those changes. Then explain each design choice.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"Walk me through a medallion pipeline" is one of the most common Databricks interview requests, and the certification's transformation domain assumes you know what belongs in each layer. Lesson 01_30 built a medallion pipeline with plain DataFrame logic. This lesson shows how Delta features make each layer reliable, incremental and auditable.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Layers and the Delta features that support them</b><br>
| Layer | Contains | Write pattern | Delta features used |<br>|---|---|---|---|<br>| <b>Bronze</b> | Raw data as received, plus audit columns (file, load time, batch id) | Append only | Schema evolution (<code>mergeSchema</code>), time travel for replay |<br>| <b>Silver</b> | Cleaned, typed, deduplicated, validated entities (one row per business key) | <code>MERGE</code> (upsert) | <code>MERGE</code>, constraints, CDF enabled for downstream |<br>| <b>Gold</b> | Business aggregates and marts | Incremental <code>MERGE</code> from Silver changes, or full rebuild | CDF consumption, liquid clustering |
</div>

```
landing (volume)  ──append──▶  BRONZE  ──clean + dedupe + MERGE──▶  SILVER ──CDF──▶  GOLD
JSON order events              orders_bronze                         orders_silver    daily_sales_gold
                               (raw + audit)                         (constraints,    (incremental
                                                                      CDF on)          MERGE)
                                                         └── invalid rows ──▶ quarantine
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A food-delivery company receives order events from its app every few minutes: created, updated (tip added), cancelled. Bronze keeps every event forever for audit. Silver holds one row per order with its latest state, validated by constraints. Gold holds revenue per restaurant per day, updated from Silver's change feed so the dashboard refreshes in seconds instead of rebuilding from billions of rows.
</div>

## Setup: schema, volume and the landing zone

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the tables' schema, a volume for landing files, and a helper to drop a new JSON batch into it.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The landing path.
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {DB}.files")
LANDING = "/Volumes/workspace/delta_lake/files/lesson13/landing"

In [0]:
import os, json, shutil
from pyspark.sql import functions as F, Window

BRONZE, SILVER, QUARANTINE, GOLD = (f"{DB}.orders_bronze", f"{DB}.orders_silver",
                                    f"{DB}.orders_quarantine", f"{DB}.daily_sales_gold")
for t in [BRONZE, SILVER, QUARANTINE, GOLD]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
shutil.rmtree(LANDING, ignore_errors=True)

def land_batch(batch_id, events):
    """Write one JSON Lines file into the landing folder, like an upstream system would."""
    os.makedirs(LANDING, exist_ok=True)
    with open(f"{LANDING}/orders_{batch_id:03d}.json", "w") as f:
        f.write("\n".join(json.dumps(e) for e in events) + "\n")
print("landing folder:", LANDING)

## 1. Bronze: append everything, with audit columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines the Bronze load: read new landing files, keep every field (as strings, so nothing is rejected), add the source file, load time and batch id, and append.<br><br>
<b>Why it matters</b><br>Bronze is the replayable record of what arrived. Append-only with audit columns means you can always answer "where did this row come from?" and rebuild Silver if its logic changes. In production, <b>Auto Loader</b> tracks which files are new (folder 08). Here, the batch id stands in for it.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After the first batch: 5 Bronze rows with <code>source_file</code>, <code>ingested_at</code> and <code>batch_id</code>.
</div>

In [0]:
EVENT_SCHEMA = "order_id STRING, restaurant STRING, status STRING, amount STRING, event_ts STRING"

def load_bronze(batch_id):
    raw = (spark.read.schema(EVENT_SCHEMA).json(f"{LANDING}/orders_{batch_id:03d}.json")
           .withColumns({"source_file": F.col("_metadata.file_name"),
                         "ingested_at": F.current_timestamp(),
                         "batch_id": F.lit(batch_id)}))
    raw.write.mode("append").option("mergeSchema", "true").saveAsTable(BRONZE)
    return raw.count()

land_batch(1, [
    {"order_id": "1001", "restaurant": "Spice Hub", "status": "created", "amount": "24.50", "event_ts": "2024-11-01 12:00:00"},
    {"order_id": "1002", "restaurant": "Spice Hub", "status": "created", "amount": "18.00", "event_ts": "2024-11-01 12:05:00"},
    {"order_id": "1003", "restaurant": "Pasta Place", "status": "created", "amount": "31.00", "event_ts": "2024-11-01 12:10:00"},
    {"order_id": "1004", "restaurant": "Pasta Place", "status": "created", "amount": "-5.00", "event_ts": "2024-11-01 12:20:00"},
    {"order_id": "1001", "restaurant": "Spice Hub", "status": "delivered", "amount": "27.50", "event_ts": "2024-11-01 12:45:00"},
])
print("bronze rows added:", load_bronze(1))
spark.table(BRONZE).show(truncate=False)

## 2. Silver: clean, deduplicate, validate, MERGE

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the Silver table with constraints and CDF enabled, then defines the Silver load: type the Bronze rows of a batch, keep the latest event per order, quarantine invalid rows, and <code>MERGE</code> the valid ones.<br><br>
<b>Why it matters</b><br>Silver is the trusted, one-row-per-entity layer. <code>MERGE</code> makes the load idempotent and handles updates (an order going from <code>created</code> to <code>delivered</code>). Constraints guarantee no other writer can break the rules. CDF lets Gold consume only what changed.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After batch 1: 3 orders in Silver (order 1001 at its latest state, <code>delivered</code> with 27.50), and order 1004 in quarantine because of its negative amount.
</div>

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER} (
        order_id   BIGINT NOT NULL,
        restaurant STRING,
        status     STRING,
        amount     DECIMAL(10,2),
        event_ts   TIMESTAMP,
        order_date DATE,
        batch_id   INT
    )
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
spark.sql(f"ALTER TABLE {SILVER} ADD CONSTRAINT positive_amount CHECK (amount > 0)")
spark.sql(f"ALTER TABLE {SILVER} ADD CONSTRAINT known_status CHECK (status IN ('created', 'delivered', 'cancelled'))")

def load_silver(batch_id):
    typed = (spark.table(BRONZE).filter(F.col("batch_id") == batch_id)
             .select(F.col("order_id").try_cast("bigint").alias("order_id"),
                     "restaurant", "status",
                     F.col("amount").try_cast("decimal(10,2)").alias("amount"),
                     F.try_to_timestamp("event_ts", F.lit("yyyy-MM-dd HH:mm:ss")).alias("event_ts"),
                     "batch_id")
             .withColumn("order_date", F.to_date("event_ts")))

    latest = (typed.withColumn("rn", F.row_number().over(Window.partitionBy("order_id").orderBy(F.desc("event_ts"))))
                   .filter("rn = 1").drop("rn"))

    valid_rule = (F.col("order_id").isNotNull() & (F.col("amount") > 0)
                  & F.col("status").isin("created", "delivered", "cancelled") & F.col("event_ts").isNotNull())
    latest.filter(~valid_rule).withColumn("quarantined_at", F.current_timestamp()) \
          .write.mode("append").saveAsTable(QUARANTINE)

    latest.filter(valid_rule).createOrReplaceTempView("silver_updates")
    spark.sql(f"""
        MERGE INTO {SILVER} t
        USING silver_updates s ON t.order_id = s.order_id
        WHEN MATCHED AND s.event_ts > t.event_ts THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

load_silver(1)
spark.table(SILVER).orderBy("order_id").show()
spark.table(QUARANTINE).select("order_id", "amount", "status").show()

## 3. Gold: incremental aggregates from Silver's change feed

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds the Gold daily sales table once, then defines an incremental update that reads Silver's change feed since the last processed version, recomputes only the affected restaurant-days, and <code>MERGE</code>s them into Gold.<br><br>
<b>Why it matters</b><br>This is how Gold stays cheap as data grows: work is proportional to <b>what changed</b>, not to the size of Silver. The last processed version is stored in a small state table, which a streaming checkpoint would track automatically in production.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Gold with one row per restaurant per day: Spice Hub 45.50 (2 orders) and Pasta Place 31.00 (1 order) on 2024-11-01.
</div>

In [0]:
STATE = f"{DB}.pipeline_state"

def silver_version():
    return spark.sql(f"DESCRIBE HISTORY {SILVER} LIMIT 1").first()["version"]

def gold_from(silver_df):
    return (silver_df.filter("status != 'cancelled'")
            .groupBy("order_date", "restaurant")
            .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders")))

def build_gold_full():
    gold_from(spark.table(SILVER)).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD)
    spark.createDataFrame([("gold_daily_sales", silver_version())], "pipeline STRING, last_version BIGINT") \
         .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(STATE)

def update_gold_incremental():
    last = spark.table(STATE).first()["last_version"]
    current = silver_version()
    if current <= last:
        print("no new Silver changes")
        return
    changes = spark.read.option("readChangeFeed", "true").option("startingVersion", last + 1).table(SILVER)
    affected = changes.select("order_date", "restaurant").distinct()
    recomputed = gold_from(spark.table(SILVER).join(affected, ["order_date", "restaurant"]))
    # restaurant-days whose orders were all cancelled must disappear from Gold
    affected.join(recomputed, ["order_date", "restaurant"], "left") \
            .select("order_date", "restaurant", "revenue", "orders").createOrReplaceTempView("gold_updates")
    spark.sql(f"""
        MERGE INTO {GOLD} g USING gold_updates u
          ON g.order_date = u.order_date AND g.restaurant = u.restaurant
        WHEN MATCHED AND u.revenue IS NULL THEN DELETE
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED AND u.revenue IS NOT NULL THEN INSERT *
    """)
    spark.sql(f"UPDATE {STATE} SET last_version = {current}")
    print(f"processed Silver versions {last + 1}..{current}, affected restaurant-days: {affected.count()}")

build_gold_full()
spark.table(GOLD).orderBy("order_date", "restaurant").show()

## 4. The next batch flows through all layers

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lands a second batch (a new order, a cancellation, a late duplicate and a malformed row), then runs Bronze, Silver and the incremental Gold update.<br><br>
<b>Why it matters</b><br>This is the steady state of a medallion pipeline: each layer processes only the new batch, and Gold is updated from Silver's changes alone. Re-running a batch's Silver <code>MERGE</code> wouldn't create duplicates.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Silver: order 1002 is now <code>cancelled</code>, and order 1005 is added. The late, older event for 1001 is ignored because the <code>MERGE</code> only applies newer events. The malformed row goes to quarantine. Gold: Spice Hub on 2024-11-01 drops to 27.50 (1 order), and Pasta Place on 2024-11-02 appears with 42.00.
</div>

In [0]:
land_batch(2, [
    {"order_id": "1002", "restaurant": "Spice Hub", "status": "cancelled", "amount": "18.00", "event_ts": "2024-11-01 12:30:00"},
    {"order_id": "1005", "restaurant": "Pasta Place", "status": "created", "amount": "42.00", "event_ts": "2024-11-02 19:00:00"},
    {"order_id": "1001", "restaurant": "Spice Hub", "status": "created", "amount": "24.50", "event_ts": "2024-11-01 12:00:00"},
    {"order_id": "abc", "restaurant": "Nowhere", "status": "created", "amount": "10.00", "event_ts": "not a time"},
])
print("bronze rows added:", load_bronze(2))
load_silver(2)
update_gold_incremental()

spark.table(SILVER).orderBy("order_id").show()
spark.table(GOLD).orderBy("order_date", "restaurant").show()

## 5. Prove it: incremental Gold equals a full rebuild, and every row is traceable

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Compares the incrementally maintained Gold with a full recompute, and traces one Silver row back to its Bronze source file.<br><br>
<b>Why it matters</b><br>These are the checks you'd run (and describe in an interview) to show the pipeline is correct: incremental logic matches the simple full logic, and lineage is preserved through the layers.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>incremental Gold matches full rebuild: True</code>, and the Bronze rows for order 1002, with their source files and batch ids.
</div>

In [0]:
print("incremental Gold matches full rebuild:",
      sorted(spark.table(GOLD).collect()) == sorted(gold_from(spark.table(SILVER)).collect()))

spark.table(BRONZE).filter("order_id = '1002'").select("order_id", "status", "event_ts", "source_file", "batch_id").show(truncate=False)

## 6. The history of each layer

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the operations recorded for each table.<br><br>
<b>Why it matters</b><br>Every layer is auditable and recoverable: Bronze shows appends, Silver shows <code>MERGE</code>s, and Gold shows the full build followed by incremental <code>MERGE</code>s. Any bad batch can be investigated with time travel and undone with <code>RESTORE</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Bronze: two <code>WRITE</code> (append) operations. Silver: create, constraints, two <code>MERGE</code>s. Gold: a create and one <code>MERGE</code>.
</div>

In [0]:
for t in [BRONZE, SILVER, GOLD]:
    print(t)
    spark.sql(f"DESCRIBE HISTORY {t}").select("version", "operation").orderBy("version").show(truncate=False)

## Design decisions to explain in an interview

| Decision | Why |
|---|---|
| Bronze stores strings and appends only | Nothing is ever rejected or lost. Silver can be rebuilt from Bronze |
| Audit columns (`source_file`, `ingested_at`, `batch_id`) | Lineage, debugging, and reprocessing a single batch |
| Silver deduplicates the batch before `MERGE` | `MERGE` needs one source row per key |
| `WHEN MATCHED AND s.event_ts > t.event_ts` | Late, older events can't overwrite newer state |
| Quarantine + constraints | Bad rows are kept for investigation, and the table rules hold for every writer |
| CDF on Silver, incremental `MERGE` into Gold | Gold work is proportional to the changes, not the table size |
| Pipeline state (`last_version`) | Exactly which changes were processed. Streaming checkpoints do this automatically |

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ What you'd change in production</b><br>
Auto Loader for Bronze (incremental file discovery and schema evolution), Structured Streaming or Lakeflow Declarative Pipelines for Silver and Gold (checkpoints, expectations, automatic orchestration), liquid clustering on the query keys, and a Lakeflow Job to schedule it all. Those come in folders 07 and 08.
</div>

## Under the hood

```
Bronze   append (mergeSchema)                        -> 1 commit per batch
Silver   read Bronze batch -> dedupe (window) -> split valid/invalid
         MERGE valid rows                            -> 1 commit, CDF records inserts/updates
Gold     read Silver CDF since last_version -> affected keys -> recompute -> MERGE
                                                    -> 1 commit, state table updated
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> dedupe (window), `MERGE` (join) and aggregation shuffle. Keep each batch small.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> each layer's write is one atomic commit, so a failure in Silver leaves Gold untouched, and the batch can be re-run.

In [0]:
spark.read.option("readChangeFeed", "true").option("startingVersion", 1).table(SILVER) \
    .groupBy("_commit_version", "_change_type").count().orderBy("_commit_version", "_change_type").show()

In [0]:
# Clean up this lesson's tables (the landing files are kept for the practice problem)
for t in [BRONZE, SILVER, QUARANTINE, GOLD, STATE]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: Silver loses history needed later</b><br>
Silver keeps only the latest state. Keep full history in Bronze, or add an SCD2 table (lesson 07).<br><br>
<b>⛔ Problem: old events overwrite newer Silver rows</b><br>The <code>MERGE</code> updates unconditionally. Compare event timestamps in the matched condition.<br><br>
<b>⛔ Problem: Gold drifts from Silver</b><br>An incremental bug. Periodically compare Gold with a full recompute (section 5), and rebuild if needed.<br><br>
<b>⛔ Problem: one bad row fails the Silver load</b><br>The constraint rejected the batch. Validate and quarantine before the <code>MERGE</code>, as in section 2.<br><br>
<b>⛔ Problem: re-running a batch doubles Bronze</b><br>Bronze is append-only. Make the Bronze load idempotent per batch (skip already-loaded files, as Auto Loader does) or deduplicate in Silver.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. Walk me through a medallion pipeline you'd build on Databricks.</b><br>
Raw files land in a volume or cloud storage. Bronze ingests them append-only, with Auto Loader in production, keeping raw values plus audit columns such as source file, load time and batch. Silver casts and cleans the data, deduplicates to the latest record per business key, validates it (quarantining bad rows), and upserts with <code>MERGE</code> into a table with constraints and change data feed enabled. Gold aggregates Silver into business marts, updated incrementally from Silver's change feed with <code>MERGE</code>. Each layer is a Delta table, so every step is atomic, auditable and restorable.<br><br>

<b>🎤 2. What belongs in Bronze vs Silver vs Gold?</b><br>
Bronze holds raw data as received, with audit columns, append-only. Silver holds cleaned, typed, deduplicated and validated entities, one row per key. Gold holds aggregated, business-level tables for reporting and serving.<br><br>

<b>🎤 3. How do you make each layer idempotent?</b><br>
Bronze tracks which files were loaded (Auto Loader checkpoints). Silver uses <code>MERGE</code> on the business key with a deduplicated source and timestamp conditions. Gold uses <code>MERGE</code> on its grain, or recomputes affected partitions, and processing state (versions or checkpoints) records what's done.<br><br>

<b>🎤 4. Why use change data feed between Silver and Gold?</b><br>
So Gold processes only the rows that changed since its last run, instead of rescanning all of Silver, which keeps the cost proportional to the changes.<br><br>

<b>🎤 5. How do you handle late or out-of-order events?</b><br>
Keep event timestamps, deduplicate by latest event time, and only update Silver when the incoming event is newer than the stored one. Gold is then recomputed for the affected keys, including past dates.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. In which layer of the medallion architecture should raw data be stored as received, with ingestion metadata, so that downstream tables can be rebuilt?</b><br>
A. Gold<br>
B. Silver<br>
C. Bronze<br>
D. Platinum<br>
<details><summary><b>Show answer</b></summary><b>C.</b></details><br><br>

<b>Q2. A Gold table aggregates a 3 billion row Silver table every hour, and the job is getting slow. Silver receives a few thousand changes per hour. Which approach is most efficient?</b><br>
A. Rebuild Gold from scratch with a bigger cluster<br>
B. Enable change data feed on Silver and update only the affected Gold rows with <code>MERGE</code><br>
C. Cache the Silver table<br>
D. Convert Silver to CSV<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Re-run the setup and batches 1 and 2, then land a batch 3 in which order 1005 is <code>delivered</code> with a 5.00 tip (amount 47.00) and a new restaurant gets its first order</li><li>Run all three layers and confirm that Gold still matches a full rebuild</li><li>Re-run <code>load_silver(3)</code> a second time and show that Silver doesn't change (idempotency)</li><li>Add a Gold table <code>restaurant_totals</code> (revenue per restaurant over all days) and update it incrementally from Silver's change feed as well</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own. Run the setup cell and sections 1–4 again first, because the previous cell dropped the tables.
</div>

In [0]:
# Rebuild the pipeline state (setup, batches 1 and 2)
for t in [BRONZE, SILVER, QUARANTINE, GOLD, STATE]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
spark.sql(f"""CREATE TABLE {SILVER} (order_id BIGINT NOT NULL, restaurant STRING, status STRING, amount DECIMAL(10,2),
              event_ts TIMESTAMP, order_date DATE, batch_id INT) TBLPROPERTIES (delta.enableChangeDataFeed = true)""")
spark.sql(f"ALTER TABLE {SILVER} ADD CONSTRAINT positive_amount CHECK (amount > 0)")
for b in [1, 2]:
    load_bronze(b); load_silver(b)
build_gold_full()

# 1-2. Batch 3 through all layers
land_batch(3, [
    {"order_id": "1005", "restaurant": "Pasta Place", "status": "delivered", "amount": "47.00", "event_ts": "2024-11-02 19:40:00"},
    {"order_id": "1006", "restaurant": "Taco Town", "status": "created", "amount": "15.00", "event_ts": "2024-11-02 20:00:00"},
])
load_bronze(3); load_silver(3); update_gold_incremental()
print("Gold matches full rebuild:", sorted(spark.table(GOLD).collect()) == sorted(gold_from(spark.table(SILVER)).collect()))

# 3. Idempotency: the same batch again changes nothing
before = sorted(spark.table(SILVER).collect())
load_silver(3)
print("Silver unchanged after re-running batch 3:", before == sorted(spark.table(SILVER).collect()))

# 4. A second incremental Gold table with its own state
TOTALS, TOTALS_STATE = f"{DB}.restaurant_totals", f"{DB}.totals_state"
def totals_from(df):
    return df.filter("status != 'cancelled'").groupBy("restaurant").agg(F.sum("amount").alias("revenue"))
totals_from(spark.table(SILVER)).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TOTALS)
spark.createDataFrame([(silver_version(),)], "last_version BIGINT").write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TOTALS_STATE)

land_batch(4, [{"order_id": "1007", "restaurant": "Taco Town", "status": "created", "amount": "9.00", "event_ts": "2024-11-03 13:00:00"}])
load_bronze(4); load_silver(4)
last = spark.table(TOTALS_STATE).first()["last_version"]
affected = spark.read.option("readChangeFeed", "true").option("startingVersion", last + 1).table(SILVER).select("restaurant").distinct()
totals_from(spark.table(SILVER).join(affected, "restaurant")).createOrReplaceTempView("totals_updates")
spark.sql(f"MERGE INTO {TOTALS} t USING totals_updates u ON t.restaurant = u.restaurant WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
spark.sql(f"UPDATE {TOTALS_STATE} SET last_version = {silver_version()}")
spark.table(TOTALS).orderBy("restaurant").show()

for t in [BRONZE, SILVER, QUARANTINE, GOLD, STATE, TOTALS, TOTALS_STATE]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
shutil.rmtree(LANDING, ignore_errors=True)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Bronze: raw, append-only, audit columns, the replayable source of truth</li><li>Silver: typed, deduplicated, validated, one row per key, maintained with <code>MERGE</code>, constraints, CDF on</li><li>Gold: business aggregates, updated incrementally from Silver's change feed with <code>MERGE</code></li><li>Idempotency at every layer: tracked files, <code>MERGE</code> on keys, stored versions or checkpoints</li><li>Quarantine invalid rows. Constraints protect the tables from every writer</li><li>Verify: incremental Gold equals a full rebuild, and lineage from Gold to Bronze is traceable</li></ul>
</div>

| Layer | Key code |
|---|---|
| Bronze | `df.withColumns({"source_file": F.col("_metadata.file_name"), ...}).write.mode("append")` |
| Silver | `row_number()` dedupe → `MERGE INTO silver ... WHEN MATCHED AND s.ts > t.ts THEN UPDATE` |
| Gold | `readChangeFeed` since the last version → affected keys → recompute → `MERGE INTO gold` |

## Git commit

```
git add 03_delta_lake/13_medallion_overview.ipynb
git commit -m "add 03_13 medallion overview notebook"
```